# SemanticCTA — Paper Figures

Each cell is **self-contained** and independently generates one PDF figure.
Run any single cell to produce its figure — no need to run other cells first.

Output: `69e83eed34d3e95c60ad4604/figs/*.pdf` at 300 DPI.

---
## Figure 2 — Cross-column attention heatmap (Method section, supports Q3)

In [ ]:
# %% Figure 2 – Attention Heatmap – self-contained
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, numpy as np, os
from matplotlib.patches import Rectangle
import matplotlib.gridspec as gridspec

plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.08,
    'axes.spines.top': False, 'axes.spines.right': False,
})
OUT = '69e83eed34d3e95c60ad4604/figs'
C = '#2166AC'; R = '#B2182B'

col_names = ['product', 'price', 'currency', 'discount', 'category', 'brand']
table_data = [
    ['iPhone 15',  '999.00', 'USD', '10%', 'Electronics', 'Apple'],
    ['Galaxy S24', '849.99', 'USD', '15%', 'Electronics', 'Samsung'],
    ['MacBook Air','1199.00','USD', '5%',  'Computers',   'Apple'],
    ['AirPods Pro','249.00', 'USD', '20%', 'Audio',       'Apple'],
]
attn = np.array([
    [0.22,0.12,0.07,0.09,0.26,0.24],
    [0.09,0.17,0.33,0.28,0.07,0.06],
    [0.08,0.38,0.22,0.16,0.07,0.09],
    [0.10,0.32,0.17,0.22,0.09,0.10],
    [0.25,0.08,0.06,0.12,0.24,0.25],
    [0.24,0.10,0.07,0.09,0.26,0.24],
])
N = 6; target = 1

fig, (ax_t, ax_h) = plt.subplots(
    1, 2, figsize=(10.5, 4.8),
    gridspec_kw={'width_ratios': [1.2, 1], 'wspace': 0.35})

# (a) table – manual rectangles
ax_t.axis('off')
ax_t.set_title('(a) Example table (target: price)',
               fontsize=13, fontweight='bold', loc='left', pad=15)
col_w = [1.3, 0.9, 0.9, 0.9, 1.2, 0.9]
row_h = 0.55; header_h = 0.65
xc = 0
for ci, (name, w) in enumerate(zip(col_names, col_w)):
    fc = C if ci == target else '#E8E8E8'
    tc = 'white' if ci == target else 'black'
    rect = Rectangle((xc, 0), w, header_h, facecolor=fc, edgecolor='#BBB', lw=0.8)
    ax_t.add_patch(rect)
    ax_t.text(xc + w/2, header_h/2, name, ha='center', va='center',
              fontsize=11, fontweight='bold', color=tc)
    xc += w
total_w = xc
for ri, row in enumerate(table_data):
    y0 = -(ri+1) * row_h; xc = 0
    for ci, (val, w) in enumerate(zip(row, col_w)):
        fc = '#D1E5F0' if ci == target else 'white'
        rect = Rectangle((xc, y0), w, row_h, facecolor=fc, edgecolor='#DDD', lw=0.5)
        ax_t.add_patch(rect)
        ax_t.text(xc + w/2, y0 + row_h/2, val, ha='center', va='center',
                  fontsize=10, color='#333')
        xc += w
ax_t.set_xlim(-0.2, total_w + 0.2)
ax_t.set_ylim(-len(table_data)*row_h - 0.3, header_h + 0.3)
ax_t.set_aspect('equal')

# (b) heatmap
ax_h.set_title('(b) Cross-column attention',
               fontsize=13, fontweight='bold', loc='left', pad=15)
im = ax_h.imshow(attn, cmap='Blues', vmin=0, vmax=0.42, aspect='equal')
for i in range(N):
    for j in range(N):
        v = attn[i, j]
        ax_h.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=10,
                  color='white' if v > 0.22 else '#333')
rect = Rectangle((-0.5, target-0.5), N, 1, lw=2.5,
                 edgecolor=R, facecolor='none', ls='--', zorder=5)
ax_h.add_patch(rect)
ax_h.set_xticks(range(N))
ax_h.set_xticklabels(col_names, rotation=35, ha='right', fontsize=10)
ax_h.set_yticks(range(N))
ax_h.set_yticklabels(col_names, fontsize=10)
ax_h.set_xlabel('Key column')
ax_h.set_ylabel('Query column')
fig.colorbar(im, ax=ax_h, shrink=0.8).set_label('Attention weight', fontsize=11)
fig.tight_layout(pad=1.5)
fig.savefig(f'{OUT}/fig-attention-heatmap.pdf')
plt.close()
print(f'Saved {OUT}/fig-attention-heatmap.pdf')

---
## Figure 3 — In-decoder embedding extraction ablation (Table 2a visualisation)

In [ ]:
# %% Figure 3 – Embedding ablation – self-contained
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, numpy as np, os

plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.08,
    'axes.spines.top': False, 'axes.spines.right': False,
})
OUT = '69e83eed34d3e95c60ad4604/figs'
C = '#2166AC'

strategies = ['D1: Final\nlayer', 'D2: Mean\npool', 'D3: L-4\n(Ours)',
              'D4: w/o\nprefix', 'D5: w/o\nprofiling']
datasets   = ['GitTablesDB', 'GitTablesSC', 'SOTAB-CTA', 'VizNet']
data = np.array([
    [72.4,69.8,66.3,61.2],
    [77.6,74.2,70.8,66.5],
    [80.3,77.1,73.5,68.9],
    [71.8,68.3,64.7,59.6],
    [78.5,75.3,71.9,67.2],
])
palette = ['#8DA0CB','#E78AC3','#A6D854','#FFD92F']
x = np.arange(5); w = 0.19

fig, ax = plt.subplots(figsize=(8.5, 4.2))
for i, (ds, clr) in enumerate(zip(datasets, palette)):
    ax.bar(x + i*w - 1.5*w, data[:,i], w, label=ds, color=clr, edgecolor='white', lw=0.6)
ax.axvspan(1.65, 2.35, color=C, alpha=0.07)
ax.text(2, 85, 'Ours', fontsize=12, fontweight='bold', color=C, ha='center')
for i in range(4):
    bx = 2 + i*w - 1.5*w
    ax.text(bx, data[2,i]+0.5, f'{data[2,i]:.1f}', ha='center', fontsize=9, color='#333')
ax.set_ylabel('Macro-F1'); ax.set_xticks(x); ax.set_xticklabels(strategies, fontsize=10)
ax.set_ylim(55, 88); ax.legend(loc='upper right', ncol=2, framealpha=0.9)
ax.yaxis.grid(True, ls='--', alpha=0.3); ax.set_axisbelow(True)
fig.tight_layout(pad=1.2)
fig.savefig(f'{OUT}/fig-embedding-ablation.pdf')
plt.close()
print(f'Saved {OUT}/fig-embedding-ablation.pdf')

---
## Figure 4 — Column relation ablation (Table 3, horizontal bar chart)

In [ ]:
# %% Figure 4 – Relation ablation – self-contained
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, numpy as np, os

plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.08,
    'axes.spines.top': False, 'axes.spines.right': False,
})
OUT = '69e83eed34d3e95c60ad4604/figs'
C = '#2166AC'; R = '#B2182B'

variants = ['SemanticCTA (Full)', 'w/o profiling text', 'Shuffled column order',
            'Masked attn to neighbors', 'w/o non-target columns', 'w/o table prefix']
scores = [84.1, 82.8, 82.3, 79.1, 78.6, 75.2]
colors = [C, R, R, R, R, '#D6604D']

fig, ax = plt.subplots(figsize=(7.5, 4.0))
y = np.arange(len(variants))
bars = ax.barh(y, scores, color=colors, edgecolor='white', lw=0.6, height=0.55)
for bar, s in zip(bars, scores):
    ax.text(s+0.3, bar.get_y()+bar.get_height()/2, f'{s:.1f}',
            va='center', fontsize=10.5, color='#333')
ax.axvline(scores[0], color=C, ls='--', alpha=0.35, lw=0.9)
for i in range(1, len(scores)):
    drop = scores[0] - scores[i]
    ax.text(scores[0]-drop/2, i, f'-{drop:.1f}', ha='center', va='center',
            fontsize=9.5, color='#555',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='#CCC', alpha=0.85, lw=0.4))
ax.set_xlabel('Macro-F1 (GitTablesSC)')
ax.set_yticks(y); ax.set_yticklabels(variants, fontsize=10.5)
ax.invert_yaxis(); ax.set_xlim(73, 87)
ax.xaxis.grid(True, ls='--', alpha=0.3); ax.set_axisbelow(True)
fig.tight_layout(pad=1.2)
fig.savefig(f'{OUT}/fig-relation-ablation.pdf')
plt.close()
print(f'Saved {OUT}/fig-relation-ablation.pdf')

---
## Figure 5 — Efficiency: Accuracy vs Latency scatter (Table 6 visualisation)

In [ ]:
# %% Figure 5 – Efficiency scatter – self-contained
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, numpy as np, os

plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.08,
    'axes.spines.top': False, 'axes.spines.right': False,
})
OUT = '69e83eed34d3e95c60ad4604/figs'
C = '#2166AC'; R = '#B2182B'

methods = [
    (12,  62.3, 'TURL',         '#888888', '^'),
    (22,  68.5, 'DODUO',        '#888888', 'v'),
    (32,  71.2, 'Starmie',      '#888888', 'P'),
    (42,  73.6, 'JSL-CTA',      '#888888', 'X'),
    (820, 73.2, 'Desc->Encode', R,         's'),
    (145, 81.3, 'Ours',         '#66C2A5', 'D'),
    (52,  81.3, 'Ours (cached)',C,         'o'),
]

fig, ax = plt.subplots(figsize=(6.5, 4.5))
for lat, f1, nm, cl, mk in methods[:4]:
    ax.scatter(lat, f1, marker=mk, s=80, c=cl, edgecolors='white', lw=0.8, zorder=3,
               label='Encoder baseline' if nm == 'TURL' else '')
lat, f1, nm, cl, mk = methods[4]
ax.scatter(lat, f1, marker=mk, s=100, c=cl, edgecolors='white', lw=0.8, zorder=3, label=nm)
for lat, f1, nm, cl, mk in methods[5:]:
    sz = 140 if 'cached' in nm else 110
    ax.scatter(lat, f1, marker=mk, s=sz, c=cl, edgecolors='white', lw=1.0, zorder=4, label=nm)
for lat, f1, nm, cl, mk in methods:
    if nm in ('TURL','DODUO','Starmie','JSL-CTA'):
        continue
    off = (12, 8) if 'cached' not in nm else (-10, 12)
    ha = 'left' if 'cached' not in nm else 'right'
    ax.annotate(nm, (lat, f1), textcoords='offset points', xytext=off,
                fontsize=11, color=cl, ha=ha, fontweight='bold')
ax.annotate('', xy=(60, 81.3), xytext=(810, 73.2),
            arrowprops=dict(arrowstyle='->', color='#AAA', ls='--', lw=1.0))
ax.text(400, 76.5, 'Pareto improvement', fontsize=9.5, color='#777',
        ha='center', style='italic', rotation=-8)
ax.set_xlabel('Latency (ms / table)'); ax.set_ylabel('Macro-F1')
ax.set_xlim(-20, 920); ax.set_ylim(58, 85)
ax.legend(loc='lower right', framealpha=0.92, fontsize=10)
ax.yaxis.grid(True, ls='--', alpha=0.3); ax.set_axisbelow(True)
fig.tight_layout(pad=1.2)
fig.savefig(f'{OUT}/fig-efficiency.pdf')
plt.close()
print(f'Saved {OUT}/fig-efficiency.pdf')

---
## Figure 6 — Embedding quality radar (Table 5 visualisation, appendix)

In [ ]:
# %% Figure 6 – Embedding quality radar – self-contained
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, numpy as np, os

plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.08,
    'axes.spines.top': False, 'axes.spines.right': False,
})
OUT = '69e83eed34d3e95c60ad4604/figs'
C = '#2166AC'; R = '#B2182B'

metrics = ['kNN\npurity', 'Linear-probe\nF1', 'Intra/inter\ndist (inv)', 'Desc-label\nagreement']
desc_enc = [0.723, 0.716, 1-0.682, 0.791]
semcta   = [0.841, 0.782, 1-0.514, 0.876]
N = len(metrics)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
ac = angles + angles[:1]

fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1)
ax.plot(ac, semcta+semcta[:1], 'o-', color=C, lw=2, ms=7, label='SemanticCTA', zorder=3)
ax.fill(ac, semcta+semcta[:1], color=C, alpha=0.12)
ax.plot(ac, desc_enc+desc_enc[:1], 's--', color=R, lw=1.5, ms=5.5, label='Desc + encoder', zorder=3)
ax.fill(ac, desc_enc+desc_enc[:1], color=R, alpha=0.06)
for a, v in zip(angles, semcta):
    ax.annotate(f'{v:.3f}', xy=(a, v), textcoords='offset points', xytext=(0, 10),
                fontsize=9.5, color=C, ha='center', fontweight='bold')
ax.set_xticks(angles); ax.set_xticklabels(metrics, fontsize=10.5)
ax.set_ylim(0, 1.0); ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['', '0.5', '', '1.0'], fontsize=8.5, color='#999')
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), framealpha=0.92, fontsize=10.5)
ax.grid(color='#DDD', lw=0.5)
fig.tight_layout(pad=1.5)
fig.savefig(f'{OUT}/fig-embedding-quality.pdf')
plt.close()
print(f'Saved {OUT}/fig-embedding-quality.pdf')

---
## Figure 7 — Layer sensitivity (Appendix, line plot)

In [ ]:
# %% Figure 7 – Layer sensitivity – self-contained
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, numpy as np, os

plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.08,
    'axes.spines.top': False, 'axes.spines.right': False,
})
OUT = '69e83eed34d3e95c60ad4604/figs'
C = '#2166AC'; R = '#B2182B'

layers = [0, 1, 2, 3]; labels = ['L-8', 'L-4', 'L-2', 'L-1']
git_sc = [76.8, 81.3, 78.1, 72.4]; sotab = [71.2, 77.6, 73.5, 66.3]

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot(layers, git_sc, 'o-', color=C, lw=2, ms=8, label='GitTablesSC', zorder=3)
ax.plot(layers, sotab,  's--', color=R, lw=1.6, ms=7, label='SOTAB-CTA', zorder=3)
for i, (g, s) in enumerate(zip(git_sc, sotab)):
    ax.text(i, g+0.7, f'{g:.1f}', ha='center', fontsize=10, color=C)
    ax.text(i, s-1.4, f'{s:.1f}', ha='center', fontsize=10, color=R)
ax.axvline(1, color='#888', ls=':', lw=1, alpha=0.5)
ax.text(1, 83, 'Best', fontsize=11, color=C, fontweight='bold', ha='center')
ax.set_xlabel('Extraction layer'); ax.set_ylabel('Macro-F1')
ax.set_xticks(layers); ax.set_xticklabels(labels)
ax.set_ylim(63, 85); ax.legend(framealpha=0.92, fontsize=10.5)
ax.yaxis.grid(True, ls='--', alpha=0.3); ax.set_axisbelow(True)
fig.tight_layout(pad=1.2)
fig.savefig(f'{OUT}/fig-layer-sensitivity.pdf')
plt.close()
print(f'Saved {OUT}/fig-layer-sensitivity.pdf')

---
## Figure 8 — Backbone sensitivity (Appendix, grouped bar chart)

In [ ]:
# %% Figure 8 – Backbone sensitivity – self-contained
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, numpy as np, os

plt.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 11,
    'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.08,
    'axes.spines.top': False, 'axes.spines.right': False,
})
OUT = '69e83eed34d3e95c60ad4604/figs'
C = '#2166AC'; R = '#B2182B'

backbones = ['Qwen2.5\n1.5B', 'Qwen2.5\n7B', 'LLaMA3\n8B', 'Mistral\n7B']
git_sc = [75.8, 81.3, 79.6, 78.4]; sotab = [70.2, 77.6, 75.1, 73.8]
x = np.arange(4); w = 0.32

fig, ax = plt.subplots(figsize=(6.5, 4.0))
b1 = ax.bar(x-w/2, git_sc, w, label='GitTablesSC', color=C, edgecolor='white', lw=0.6)
b2 = ax.bar(x+w/2, sotab,  w, label='SOTAB-CTA',   color=R, edgecolor='white', lw=0.6)
for bar in b1:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{bar.get_height():.1f}', ha='center', fontsize=9.5, color=C)
for bar in b2:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
            f'{bar.get_height():.1f}', ha='center', fontsize=9.5, color=R)
ax.axvspan(0.65, 1.35, color=C, alpha=0.06)
ax.text(1, 85, 'Default', fontsize=10, color=C, ha='center', style='italic')
ax.set_ylabel('Macro-F1'); ax.set_xticks(x); ax.set_xticklabels(backbones, fontsize=10.5)
ax.set_ylim(65, 87); ax.legend(loc='upper left', framealpha=0.92, fontsize=10.5)
ax.yaxis.grid(True, ls='--', alpha=0.3); ax.set_axisbelow(True)
fig.tight_layout(pad=1.2)
fig.savefig(f'{OUT}/fig-backbone-sensitivity.pdf')
plt.close()
print(f'Saved {OUT}/fig-backbone-sensitivity.pdf')